# TextGen + SuperboogaV2 for Paperspace — Distribution Version 1.0

Paperspace Gradient上で、TextGen（oobabooga/textgen）とSuperboogaV2を起動するためのワンクリックNotebookです。

このNotebookは、Paperspace GradientでGGUFモデルを使ったチャット環境を構築し、必要に応じてSuperboogaV2によるRAG（外部テキスト参照）を利用できるようにすることを目的としています。

---

## 対象環境

このNotebookは、以下の環境を主な対象として作成しています。

```text
Platform:
  Paperspace Gradient Notebook

Recommended Container:
  shibashiba2/paperspace-gradient-base-py313-pytorch2.8.0:v11.0

Expected Runtime:
  Python 3.13
  PyTorch 2.8.x
  CUDA-enabled GPU environment

Recommended GPU:
  NVIDIA RTX A6000 48GB class or higher for large GGUF models
```

上記以外の環境でも動作する可能性はありますが、Python、CUDA、PyTorch、pip依存関係の差により追加調整が必要になる場合があります。

---

## 主な機能

```text
TextGenを /notebooks/textgen にセットアップ
GGUF / llama.cpp loader依存関係を確認
SuperboogaV2を有効化
SuperboogaV2のEmbedding実行先をCPU固定
不足依存関係を自動チェック/修復
Gradio share URLによる外部アクセス
7860番ポート競合の検出と回避
古いTextGen/GPUプロセスの検出と整理
起動前VRAMチェック
モデル自動アンロードをデフォルト無効化
通常ログの静音化
```

---

## 含まれないもの

このNotebookには以下は含まれていません。

```text
GGUFモデル本体
有料APIキー
Hugging Faceの認証情報
作品データや個人データ
TextGen公式リポジトリの恒久的な改変
```

モデルファイルのライセンス、利用条件、再配布条件は、各モデルの配布元で確認してください。

---

## モデルファイルの配置

GGUFモデルは、以下のディレクトリに配置してください。

```text
/notebooks/textgen/user_data/models
```

一時的に `/tmp/models` に置いたモデルは、コピーせず `tmp__` から始まるsymlinkとして表示されます。  
大容量GGUFを二重コピーしないための設計です。

---

## 基本の使い方

```text
1. Paperspace Gradientで推奨コンテナを選択する
2. このNotebookを開く
3. 起動セルを上から1回実行する
4. ログに表示される Gradio public URL を開く
5. TextGenのModelタブでGGUFモデルを選択してLoadする
6. Chatタブで会話を開始する
7. SuperboogaV2を使う場合は、拡張タブからテキスト・URL・ファイルを投入する
```

起動ログの先頭に以下が出れば、この配布バージョンです。

```text
=== TextGen + SuperboogaV2 Paperspace Distribution v1.0 ===
```

---

## SuperboogaV2の簡易テスト

SuperboogaV2に以下のような架空情報を投入します。

```text
テスト識別子: 紫電カレー7264
紫電カレー7264とは、木星の羊羹職人が使う架空の合言葉である。
```

その後、新規チャットで以下を質問します。

```text
紫電カレー7264とは何？
```

投入した説明に近い内容を返せば、SuperboogaV2のRAGが機能しています。

---

## セキュリティ注意

このNotebookはデフォルトで `--share` を使用します。  
Gradioの公開URLは、URLを知っている第三者がアクセスできる可能性があります。

公開URL上では、以下の取り扱いに注意してください。

```text
個人情報
業務秘密
未公開作品
APIキー
ログイン情報
機密資料
```

認証を追加したい場合は、起動セル内の `EXTRA_FLAGS` にTextGen/Gradioの認証フラグを追加してください。

例:

```python
EXTRA_FLAGS = ["--gradio-auth", "user:password"]
```

---

## GPUプロセス整理について

配布バージョンでは、安全のため aggressive GPU kill はデフォルトOFFです。

```python
AGGRESSIVE_KILL_GPU_PROCESSES = False
```

Paperspace上でTextGen専用にGPUを使用しており、カーネル再起動後もVRAMが解放されない場合のみ、一時的に `True` にしてください。

```python
AGGRESSIVE_KILL_GPU_PROCESSES = True
```

共有GPU、複数作業、別プロセスが動いている環境では、安易にONにしないでください。


## 設定ガイド

通常は起動セル上部の設定だけ確認すれば運用できます。

---

## 基本設定

通常は変更不要です。

```python
REPO_DIR = Path("/notebooks/textgen")
PERSISTENT_MODELS_DIR = REPO_DIR / "user_data" / "models"
ENABLE_SUPERBOOGAV2 = True
FORCE_SUPERBOOGA_EMBEDDING_CPU = True
MODEL_IDLE_TIMEOUT_MINUTES = None
```

### `MODEL_IDLE_TIMEOUT_MINUTES`

デフォルトでは、放置してもモデルを自動アンロードしません。

```python
MODEL_IDLE_TIMEOUT_MINUTES = None
```

VRAM節約のため、一定時間無操作でモデルを降ろしたい場合のみ分数を指定します。

```python
MODEL_IDLE_TIMEOUT_MINUTES = 180
```

---

## メンテナンス設定

### `UPDATE_TEXTGEN`

TextGen本体を更新したい場合のみ `True` にします。

```python
UPDATE_TEXTGEN = False
```

更新後は `False` に戻すことを推奨します。

---

### `FORCE_RESET_REPO`

git更新がローカル変更で失敗する場合のみ使用します。

```python
FORCE_RESET_REPO = False
```

TextGen側のローカル変更を戻すため、通常は使用しません。

---

### `FORCE_REPAIR_DEPS`

依存関係を入れ直したい場合のみ使用します。

```python
FORCE_REPAIR_DEPS = False
```

通常の不足依存は自動検出されるため、基本的には `False` のままで問題ありません。

---

## SuperboogaV2設定

### `FORCE_SUPERBOOGA_EMBEDDING_CPU`

大容量GGUFモデルとEmbedding処理のVRAM競合を避けるため、デフォルトでCPU固定にしています。

```python
FORCE_SUPERBOOGA_EMBEDDING_CPU = True
SUPERBOOGA_EMBEDDING_DEVICE = "cpu"
```

---

### `USE_MULTILINGUAL_RAG_PATCH`

日本語を含む多言語RAGを強めたい場合のみ使用します。

```python
USE_MULTILINGUAL_RAG_PATCH = False
```

通常はOFFで問題ありません。

---

## ログ設定

通常時は不足モジュール検出やpip installのログを短くします。

```python
QUIET_MISSING_MODULE_CHECKS = True
PIP_QUIET_INSTALL = True
SHOW_FEATURE_SUMMARY = True
```

pip installが失敗した場合は、詳細ログつきで再実行されます。

---

## 追加起動フラグ

TextGenの起動フラグを追加したい場合は `EXTRA_FLAGS` を使用します。

```python
EXTRA_FLAGS = []
```

例: Gradio認証を追加する場合

```python
EXTRA_FLAGS = ["--gradio-auth", "user:password"]
```

---

## 起動後の確認

正常起動すると、ログに以下のようなURLが表示されます。

```text
Running on public URL: https://xxxxx.gradio.live
```

URLを開き、ModelタブでGGUFモデルをLoadしてください。


In [ ]:
# ============================================================
# 🔥 TextGen + SuperboogaV2 for Paperspace — Distribution Version 1.0
# ============================================================
# Public distribution Notebook for Paperspace Gradient.
#
# Purpose:
# - Set up and launch oobabooga/textgen under /notebooks/textgen.
# - Enable SuperboogaV2 for RAG-style retrieval.
# - Avoid VRAM conflicts by forcing SuperboogaV2 embeddings to CPU by default.
# - Provide safer defaults for public distribution.
#
# Notes:
# - GGUF model files are not included.
# - Gradio share URLs are accessible to anyone who has the URL.
# - Review settings below before running in shared or sensitive environments.
# ============================================================

# 配布用ワンクリック起動セル。
# 初心者でも使えるよう、設定・安全注意・トラブル対策をコメントに明記している。
# 通常はこのセルだけ実行すればOK。
# 
# 重要:
# - モデルファイルは同梱していない。各自でGGUFを配置する。
# - Gradio share URLは、URLを知っている人がアクセスできる。
# - SuperboogaV2のEmbeddingはCPU固定。大きいGGUFモデルとのVRAM競合を避けるため。
# ============================================================

from pathlib import Path
import os
import sys
import subprocess
import shlex
import json
import shutil
import platform
import socket
import time
import signal

BOOTSTRAP_VERSION = "TextGen + SuperboogaV2 Paperspace Distribution v1.0"

# ============================================================
# Distribution metadata
# ============================================================
DISTRIBUTION_VERSION = "1.0"
DISTRIBUTION_PROFILE = "paperspace-public-v1.0"
RECOMMENDED_CONTAINER_IMAGE = "shibashiba2/paperspace-gradient-base-py313-pytorch2.8.0:v11.0"
RECOMMENDED_PLATFORM = "Paperspace Gradient Notebook"
RECOMMENDED_PYTHON = "Python 3.13"
RECOMMENDED_TORCH = "PyTorch 2.8.x / CUDA environment"
RECOMMENDED_GPU_NOTE = "NVIDIA RTX A6000 48GB class or higher recommended for large Q8 GGUF models"

# ============================================================
# 設定
# ============================================================

# 基本はこの設定のままでOK。
# 初回配布利用者が主に見る場所。
#
# よく変更する可能性があるもの:
# - UPDATE_TEXTGEN: TextGen本体を更新したい時だけTrue
# - FORCE_REPAIR_DEPS: 依存を入れ直したい時だけTrue
# - USE_MULTILINGUAL_RAG_PATCH: 日本語RAGを強めたい時だけTrue
# - MODEL_IDLE_TIMEOUT_MINUTES: 放置時にモデルを自動アンロードしたい時だけ分数を指定
# - AGGRESSIVE_KILL_GPU_PROCESSES: VRAM残骸を強制掃除したい時だけTrue

# TextGen本体の配置先。
# 配布版では /notebooks/textgen を標準パスにする。
# 旧 /notebooks/text-generation-webui は参照しない。
REPO_DIR = Path("/notebooks/textgen")
REPO_URL = "https://github.com/oobabooga/textgen"

# モデル置き場。
# 推奨: GGUFモデルを /notebooks/textgen/user_data/models に置く。
# /tmp/models に置いた場合は、コピーせず tmp__ から始まるsymlinkとして user_data/models に見せる。
# 大きいGGUFファイルを二重コピーしないための設計。
PERSISTENT_MODELS_DIR = REPO_DIR / "user_data" / "models"
TMP_MODELS_DIR = Path("/tmp/models")

CACHE_DIR = Path("/notebooks/textgen_cache")
HF_HOME_DIR = CACHE_DIR / "huggingface"
ST_HOME_DIR = CACHE_DIR / "sentence_transformers"
PIP_CACHE_DIR = CACHE_DIR / "pip"

# 普段はFalse。
UPDATE_TEXTGEN = False

# git更新がローカル変更で失敗する時だけTrue。
FORCE_RESET_REPO = False

# 依存を全部入れ直したい時だけTrue。
FORCE_REPAIR_DEPS = False

# TextGen公式ファイルをパッチして日本語RAGを強めたい時だけTrue。
USE_MULTILINGUAL_RAG_PATCH = False
MULTILINGUAL_EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# 重要: SuperboogaV2のEmbedding生成をCPU固定する。
# Gemma Q8などでVRAMがほぼ埋まると、Embeddingをcudaで動かした瞬間にOOMするため。
FORCE_SUPERBOOGA_EMBEDDING_CPU = True
SUPERBOOGA_EMBEDDING_DEVICE = "cpu"

ENABLE_SUPERBOOGAV2 = True

# TextGen起動フラグ。
# --share: Gradioの一時公開URLを発行する。URLを知っている人はアクセスできる点に注意。
# 認証を付けたい場合は、TextGen側の --gradio-auth 等をEXTRA_FLAGSへ追加する運用を推奨。
LAUNCH_FLAGS = [
    "--listen",
    "--listen-host", "0.0.0.0",
    "--listen-port", "7860",
    "--share",
    "--no-electron",
]

if ENABLE_SUPERBOOGAV2:
    LAUNCH_FLAGS += ["--extensions", "superboogav2"]

# 追加のTextGen起動フラグを入れる場所。
# 例:
# EXTRA_FLAGS = ["--gradio-auth", "user:password"]
# EXTRA_FLAGS = ["--api"]
# 分からない場合は空のままでOK。
EXTRA_FLAGS = []

# 長時間放置してもモデルを自動アンロードしない設定。
# Noneなら --idle-timeout を付けないため、TextGen起動中はモデルが基本的に常駐する。
# VRAM節約したい配布利用者だけ、分数を指定する。
# 例: MODEL_IDLE_TIMEOUT_MINUTES = 180  # 180分無操作でモデルを降ろす
MODEL_IDLE_TIMEOUT_MINUTES = None

# 通常時は不足モジュール検出で長いTracebackを出さない。
QUIET_MISSING_MODULE_CHECKS = True

# pip installログを短くする。失敗時は詳細ログつきで再試行する。
PIP_QUIET_INSTALL = True

# pip/setuptools/wheel更新は毎回ではなく、必要時だけにする。
UPGRADE_PIP_TOOLS_ON_REPAIR = False

# 起動時に有効機能一覧を表示する。
SHOW_FEATURE_SUMMARY = True

# カーネル再起動後に古いTextGenプロセスが7860を掴んだまま残ることがある。
# Trueなら /notebooks/textgen/server.py の古いプロセスだけを起動前に停止する。
AUTO_KILL_STALE_TEXTGEN = True

# 7860が埋まっている場合、7861〜7899の空きポートに自動退避する。
AUTO_SELECT_FREE_PORT = True
PORT_RANGE_START = 7860
PORT_RANGE_END = 7899

# カーネル再起動後に古い llama.cpp / llama-server 系プロセスがVRAMを掴んだまま残ることがある。
# TrueならTextGen由来らしいGPUプロセスだけを起動前に停止する。
AUTO_KILL_STALE_GPU_TEXTGEN = True

# 起動前にVRAM空き容量を表示する。古いプロセス掃除後の確認用。
SHOW_VRAM_AFTER_CLEANUP = True

# GPUの空きが少なすぎる場合、nvidia-smi上のGPU computeプロセスを強制停止するモード。
# 配布版では安全のためデフォルトFalse。
#
# Trueにする場面:
# - Paperspaceの専用GPUでTextGenだけ使っている
# - カーネル再起動後もVRAMがほぼ埋まっている
# - nvidia-smiで古い llama / textgen / server.py 由来プロセスが残っている
#
# Falseのままにする場面:
# - 同じGPUで別の学習・生成・実験を同時実行している
# - 他人とGPUを共有している
AGGRESSIVE_KILL_GPU_PROCESSES = False

# 起動前に最低限ほしい空きVRAM。A6000でQ8を読むなら20GB以上は欲しい。
MIN_FREE_VRAM_BEFORE_LAUNCH_MIB = 20000

# ============================================================
# 共通関数
# ============================================================

def log(msg=""):
    print(msg, flush=True)

def run(cmd, cwd=None, check=True, env=None):
    log("▶ " + " ".join(shlex.quote(str(x)) for x in cmd))
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=check,
        text=True,
    )

def run_capture(cmd, cwd=None, env=None):
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

def print_feature_summary():
    if not SHOW_FEATURE_SUMMARY:
        return
    log("🧭 Paperspace Distribution v1.0 feature summary")
    log(f"  - profile: {DISTRIBUTION_PROFILE}")
    log(f"  - recommended container: {RECOMMENDED_CONTAINER_IMAGE}")
    log(f"  - platform: {RECOMMENDED_PLATFORM}")
    features = [
        "idle-timeout disabled by default",
        "SuperboogaV2 embedding forced to CPU/env-controlled",
        "portable dependency self-check/repair",
        "GGUF llama.cpp loader dependency check",
        "7860 port conflict handling",
        "stale TextGen/GPU process cleanup",
        "VRAM preflight check",
        "quiet missing-module checks",
        "quiet pip install with verbose retry on failure",
        "pip/setuptools/wheel upgrade only when explicitly enabled",
        "aggressive GPU process kill is available but OFF by default",
    ]
    for item in features:
        log(f"  - {item}")

def pip_install(reqs, label="", check=False, quiet=None):
    if isinstance(reqs, str):
        reqs = [reqs]
    if quiet is None:
        quiet = PIP_QUIET_INSTALL

    reqs = list(reqs)
    base_cmd = [sys.executable, "-m", "pip", "install"]
    common_tail = reqs + ["--upgrade-strategy", "only-if-needed"]

    if label:
        log(f"\n📦 {label}")

    if quiet:
        quiet_cmd = base_cmd + ["-q"] + common_tail
        log("▶ pip install -q " + " ".join(shlex.quote(str(x)) for x in reqs))
        r = run_capture(quiet_cmd)
        if r.returncode == 0:
            log("  ✅ ok")
            return r

        log("  ⚠️ quiet pip install failed. Retrying with verbose output...")
        verbose_cmd = base_cmd + common_tail
        r2 = run(verbose_cmd, check=False)
        if check and r2.returncode != 0:
            r2.check_returncode()
        return r2

    cmd = base_cmd + common_tail
    return run(cmd, check=check)

def upgrade_pip_tools_if_enabled(reason="dependency repair"):
    if not UPGRADE_PIP_TOOLS_ON_REPAIR:
        log(f"  ℹ️ skip pip/setuptools/wheel upgrade ({reason})")
        return None
    return pip_install(["-U", "pip", "setuptools", "wheel"], label=f"pip base ({reason})", check=False, quiet=False)


def module_ok(name, import_code, cwd=None, verbose=False):
    r = run_capture([sys.executable, "-c", import_code], cwd=cwd)
    ok = r.returncode == 0
    if verbose or not ok:
        status = "OK" if ok else "MISSING"
        log(f"  {status}: {name}")
        if not ok and r.stdout:
            if QUIET_MISSING_MODULE_CHECKS and not verbose:
                tail_lines = [x.strip() for x in r.stdout.splitlines() if x.strip()]
                if tail_lines:
                    log("  hint: " + tail_lines[-1][:300])
            else:
                log("  " + r.stdout[-1200:].replace("\n", "\n  "))
    return ok


def ensure_dirs():
    for d in [CACHE_DIR, HF_HOME_DIR, ST_HOME_DIR, PIP_CACHE_DIR, PERSISTENT_MODELS_DIR, TMP_MODELS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

def setup_env():
    # TRANSFORMERS_CACHEは非推奨警告が出るので使わない。
    os.environ["HF_HOME"] = str(HF_HOME_DIR)
    os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME_DIR / "hub")
    os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(ST_HOME_DIR)
    os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["ANONYMIZED_TELEMETRY"] = "False"
    os.environ["CHROMA_TELEMETRY"] = "False"
    os.environ["POSTHOG_DISABLED"] = "true"
    os.environ["SUPERBOOGAV2_EMBEDDING_MODEL"] = MULTILINGUAL_EMBEDDING_MODEL
    os.environ["SUPERBOOGA_EMBEDDING_DEVICE"] = SUPERBOOGA_EMBEDDING_DEVICE
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================
# Repo
# ============================================================

def ensure_repo():
    if not REPO_DIR.exists():
        log(f"⬇️ TextGen clone: {REPO_URL}")
        run(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        log(f"✅ TextGen repo exists: {REPO_DIR}")

    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} is not a git repository.")

    run(["git", "remote", "set-url", "origin", REPO_URL], cwd=REPO_DIR, check=False)

    if UPDATE_TEXTGEN or FORCE_RESET_REPO:
        log("\n🔄 TextGen update")
        run(["git", "fetch", "--prune", "origin"], cwd=REPO_DIR, check=False)

        if FORCE_RESET_REPO:
            log("⚠️ FORCE_RESET_REPO=True: reset --hard origin/main")
            run(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)
        else:
            r = run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=False)
            if r.returncode != 0:
                log("⚠️ git pull failed. Use FORCE_RESET_REPO=True if needed.")
    else:
        log("ℹ️ UPDATE_TEXTGEN=False: git update skipped")

# ============================================================
# GPU
# ============================================================

def check_gpu():
    log("\n🧠 GPU check")
    r = run_capture(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"])
    if r.returncode == 0 and r.stdout.strip():
        for line in r.stdout.strip().splitlines():
            log("  " + line)
    else:
        log("  ⚠️ nvidia-smi not available")

# ============================================================
# TextGen portable dependencies
# ============================================================

TEXTGEN_PORTABLE_CHECKS = {
    "gradio": ("import gradio, gradio_client; print('gradio ok')", [
        "https://github.com/oobabooga/gradio/releases/download/4.37.2-custom.21/gradio-4.37.2+custom.21-py3-none-any.whl",
        "https://github.com/oobabooga/gradio/releases/download/4.37.2-custom.21/gradio_client-1.0.2+custom.21-py3-none-any.whl",
    ]),
    "ddgs": ("from ddgs import DDGS; print('ddgs ok')", ["ddgs==9.14.1"]),
    "fastapi": ("import fastapi; print('fastapi ok')", ["fastapi==0.112.4"]),
    "jinja2": ("import jinja2; print('jinja2 ok')", ["jinja2==3.1.6"]),
    "markdown": ("import markdown; print('markdown ok')", ["markdown"]),
    "mcp": ("import mcp; print('mcp ok')", ["mcp==1.27.0"]),
    "pydantic": ("import pydantic; print('pydantic ok')", ["pydantic==2.11.0"]),
    "pymupdf": ("import fitz; print('pymupdf ok')", ["pymupdf==1.27.*"]),
    "python-docx": ("import docx; print('docx ok')", ["python-docx==1.1.2"]),
    "pyyaml": ("import yaml; print('yaml ok')", ["pyyaml"]),
    "requests": ("import requests; print('requests ok')", ["requests"]),
    "rich": ("import rich; print('rich ok')", ["rich"]),
    "trafilatura": ("import trafilatura; print('trafilatura ok')", ["trafilatura==2.0.0"]),
    "tqdm": ("import tqdm; print('tqdm ok')", ["tqdm"]),
    "sse-starlette": ("import sse_starlette; print('sse ok')", ["sse-starlette==1.6.5"]),
    "tiktoken": ("import tiktoken; print('tiktoken ok')", ["tiktoken"]),
    "audioop-lts": ("import audioop; print('audioop ok')", ['audioop-lts<1.0; python_version >= "3.13"']),
    "huggingface-hub": ("import huggingface_hub; print('hf hub ok')", ["huggingface-hub"]),
}

def ensure_textgen_portable_deps():
    log("\n🔎 TextGen portable dependencies check")
    missing_install_reqs = []

    for name, (check_code, install_reqs) in TEXTGEN_PORTABLE_CHECKS.items():
        if FORCE_REPAIR_DEPS or not module_ok(name, check_code):
            missing_install_reqs.extend(install_reqs)

    if not missing_install_reqs and not FORCE_REPAIR_DEPS:
        log("✅ TextGen portable dependencies OK")
        return

    log("\n🛠 Repairing TextGen portable dependencies")
    upgrade_pip_tools_if_enabled("dependency repair")

    # 重複除去
    deduped = []
    seen = set()
    for req in missing_install_reqs:
        if req not in seen:
            deduped.append(req)
            seen.add(req)

    # flask_cloudflaredは入れない。過去にblinker 1.4で失敗したため。
    if deduped:
        log("  install targets:")
        for x in deduped:
            log(f"   - {x}")

    r = pip_install(deduped, label="TextGen portable deps batch", check=False)
    if r.returncode != 0:
        log("⚠️ batch install failed. Installing one by one.")
        failed = []
        for req in deduped:
            rr = pip_install(req, label=f"textgen dep: {req}", check=False)
            if rr.returncode != 0:
                failed.append(req)
        if failed:
            log("⚠️ failed installs: " + repr(failed))

    # 必須import再検査。ddgsはここで絶対に確認する。
    still_missing = []
    for name, (check_code, _) in TEXTGEN_PORTABLE_CHECKS.items():
        if not module_ok(name, check_code):
            still_missing.append(name)

    if still_missing:
        raise RuntimeError("TextGen portable dependencies still missing: " + ", ".join(still_missing))

    log("✅ TextGen portable dependencies repaired")

# ============================================================
# GGUF / llama.cpp loader
# ============================================================

def ensure_gguf_loader_deps():
    log("\n🔎 GGUF / llama.cpp loader dependencies check")
    code = "import llama_cpp_binaries; print('llama_cpp_binaries ok')"

    if module_ok("llama_cpp_binaries", code) and not FORCE_REPAIR_DEPS:
        log("✅ llama_cpp_binaries OK")
        return

    log("🛠 Installing llama_cpp_binaries")

    system = platform.system()
    machine = platform.machine().lower()

    if system == "Linux" and machine in {"x86_64", "amd64"}:
        wheel = "https://github.com/oobabooga/llama-cpp-binaries/releases/download/v0.138.0/llama_cpp_binaries-0.138.0+cu124-py3-none-linux_x86_64.whl"
    elif system == "Windows":
        wheel = "https://github.com/oobabooga/llama-cpp-binaries/releases/download/v0.138.0/llama_cpp_binaries-0.138.0+cu124-py3-none-win_amd64.whl"
    else:
        raise RuntimeError(f"Unsupported platform for llama_cpp_binaries wheel: {system} {machine}")

    r = pip_install(wheel, label="llama_cpp_binaries", check=False)
    if r.returncode != 0:
        raise RuntimeError("Failed to install llama_cpp_binaries.")

    if not module_ok("llama_cpp_binaries", code):
        raise RuntimeError("llama_cpp_binaries installed but still cannot import.")

    log("✅ llama_cpp_binaries installed")

# ============================================================
# SuperboogaV2 dependencies
# ============================================================

SUPERBOOGA_CHECKS = {
    "bs4": ("import bs4; print('bs4 ok')", ["beautifulsoup4==4.13.3"]),
    "chromadb": ("import chromadb; from chromadb.config import Settings; chromadb.Client(Settings(anonymized_telemetry=False)); print('chroma ok')", ["chromadb==0.6.3"]),
    "lxml": ("import lxml; print('lxml ok')", ["lxml"]),
    "nltk": ("import nltk; print('nltk ok')", ["nltk"]),
    "optuna": ("import optuna; print('optuna ok')", ["optuna"]),
    "pandas": ("import pandas; print('pandas ok')", ["pandas"]),
    "posthog": ("import posthog; print('posthog ok')", ["posthog==3.13.0"]),
    "sentence-transformers": ("import sentence_transformers; print('st ok')", ["sentence_transformers==3.3.1"]),
    "spacy": ("import spacy; print('spacy ok')", ["spacy"]),
    "pytextrank": ("import pytextrank; print('pytextrank ok')", ["pytextrank"]),
    "num2words": ("import num2words; print('num2words ok')", ["num2words"]),
    "fitz": ("import fitz; print('fitz ok')", ["PyMuPDF"]),
    "docx": ("import docx; print('docx ok')", ["python-docx"]),
    "pptx": ("import pptx; print('pptx ok')", ["python-pptx"]),
    "openpyxl": ("import openpyxl; print('openpyxl ok')", ["openpyxl"]),
    "odf": ("import odf; print('odf ok')", ["odfpy"]),
}

def ensure_superbooga_deps():
    if not ENABLE_SUPERBOOGAV2:
        return

    log("\n🔎 SuperboogaV2 dependencies check")
    missing_install_reqs = []

    for name, (check_code, install_reqs) in SUPERBOOGA_CHECKS.items():
        if FORCE_REPAIR_DEPS or not module_ok(name, check_code):
            missing_install_reqs.extend(install_reqs)

    if missing_install_reqs:
        log("\n🛠 Repairing SuperboogaV2 dependencies")
        upgrade_pip_tools_if_enabled("dependency repair")

        # chromadb/sentence-transformersは公式pin優先だが、失敗時fallback
        normal = []
        for req in missing_install_reqs:
            if req not in normal:
                normal.append(req)

        r = pip_install(normal, label="SuperboogaV2 deps batch", check=False)
        if r.returncode != 0:
            log("⚠️ batch install failed. Installing one by one.")
            for req in normal:
                rr = pip_install(req, label=f"superbooga dep: {req}", check=False)

        # chromadb fallback
        chroma_code = SUPERBOOGA_CHECKS["chromadb"][0]
        if not module_ok("chromadb", chroma_code):
            log("⚠️ chromadb==0.6.3 failed. Trying fallback chromadb>=1.0,<2")
            pip_install("chromadb>=1.0,<2", label="chromadb fallback", check=True)

        # sentence-transformers fallback
        st_code = SUPERBOOGA_CHECKS["sentence-transformers"][0]
        if not module_ok("sentence-transformers", st_code):
            log("⚠️ sentence_transformers==3.3.1 failed. Trying fallback.")
            pip_install("sentence-transformers>=3.3,<6", label="sentence-transformers fallback", check=True)
    else:
        log("✅ SuperboogaV2 base deps OK")

    # spaCy model
    spacy_model_code = "import spacy; spacy.load('en_core_web_sm'); print('spacy model ok')"
    if FORCE_REPAIR_DEPS or not module_ok("en_core_web_sm", spacy_model_code):
        log("📥 Installing spaCy model en_core_web_sm")
        run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

    # NLTK data
    log("📥 Ensuring NLTK data")
    import nltk
    nltk_dir = REPO_DIR / "extensions" / "superboogav2" / "nltk_data"
    nltk_dir.mkdir(parents=True, exist_ok=True)
    for pkg in [
        "stopwords",
        "wordnet",
        "omw-1.4",
        "averaged_perceptron_tagger",
        "averaged_perceptron_tagger_eng",
        "punkt",
        "punkt_tab",
    ]:
        try:
            nltk.download(pkg, download_dir=str(nltk_dir), quiet=True)
            log(f"  ✅ {pkg}")
        except Exception as e:
            log(f"  ⚠️ {pkg}: {e}")

    # extension import
    ext_code = r"""
from extensions.superboogav2.chromadb import make_collector
print('superboogav2 extension import ok')
"""
    if not module_ok("superboogav2 extension", ext_code, cwd=REPO_DIR):
        raise RuntimeError("SuperboogaV2 extension import failed.")

    log("✅ SuperboogaV2 dependencies OK")

# ============================================================
# Superbooga config
# ============================================================

def configure_superbooga():
    if not ENABLE_SUPERBOOGAV2:
        return

    super_dir = REPO_DIR / "extensions" / "superboogav2"
    if not super_dir.exists():
        raise RuntimeError(f"SuperboogaV2 folder not found: {super_dir}")

    # APIは通常OFF
    config_path = super_dir / "config.json"
    if config_path.exists():
        try:
            cfg = json.loads(config_path.read_text(encoding="utf-8"))
            if "api_on" in cfg:
                cfg["api_on"]["default"] = False
            config_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
        except Exception as e:
            log(f"⚠️ config update skipped: {e}")

    chromadb_py = super_dir / "chromadb.py"
    if not chromadb_py.exists():
        raise RuntimeError(f"chromadb.py not found: {chromadb_py}")

    text = chromadb_py.read_text(encoding="utf-8")
    backup = chromadb_py.with_suffix(".py.official_backup")
    if not backup.exists():
        shutil.copy2(chromadb_py, backup)

    changed = False

    # ------------------------------------------------------------
    # 1) Embedding device CPU固定
    # ------------------------------------------------------------
    # 公式は device=("cuda" if torch.cuda.is_available() else "cpu")。
    # これだと大きいGGUFモデルがVRAMを食った後に、SentenceTransformerまでcudaへ行きOOMする。
    if FORCE_SUPERBOOGA_EMBEDDING_CPU:
        cpu_replacements = [
            (
                'device=("cuda" if torch.cuda.is_available() else "cpu"),',
                'device=__import__("os").environ.get("SUPERBOOGA_EMBEDDING_DEVICE", "cpu"),',
            ),
            (
                "device=('cuda' if torch.cuda.is_available() else 'cpu'),",
                'device=__import__("os").environ.get("SUPERBOOGA_EMBEDDING_DEVICE", "cpu"),',
            ),
        ]

        for old, new in cpu_replacements:
            if old in text:
                text = text.replace(old, new)
                changed = True

        if 'SUPERBOOGA_EMBEDDING_DEVICE' in text:
            log(f"✅ SuperboogaV2 embedding device patch active: {SUPERBOOGA_EMBEDDING_DEVICE}")
        else:
            log("⚠️ Embedding device patch target not found. TextGen implementation may have changed.")

    # ------------------------------------------------------------
    # 2) 日本語RAG用Embeddingモデル差し替え
    # ------------------------------------------------------------
    if USE_MULTILINGUAL_RAG_PATCH:
        old = '"sentence-transformers/all-mpnet-base-v2"'
        new = '__import__("os").environ.get("SUPERBOOGAV2_EMBEDDING_MODEL", "sentence-transformers/all-mpnet-base-v2")'
        if new in text:
            log("✅ multilingual embedding model patch already active")
        elif old in text:
            text = text.replace(old, new)
            changed = True
            log(f"✅ multilingual embedding model patch: {MULTILINGUAL_EMBEDDING_MODEL}")
        else:
            log("⚠️ multilingual patch target not found")
    else:
        log("✅ SuperboogaV2 official embedding model setting")

    if changed:
        chromadb_py.write_text(text, encoding="utf-8")
        log("✅ SuperboogaV2 chromadb.py patched")
    else:
        log("ℹ️ SuperboogaV2 chromadb.py unchanged")


# ============================================================
# Models: /tmp/models -> user_data/models/tmp__
# ============================================================

def link_tmp_models():
    log("\n🔗 Linking /tmp/models into user_data/models")
    PERSISTENT_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    TMP_MODELS_DIR.mkdir(parents=True, exist_ok=True)

    prefix = "tmp__"

    removed = 0
    for p in PERSISTENT_MODELS_DIR.iterdir():
        if p.name.startswith(prefix) and p.is_symlink():
            p.unlink()
            removed += 1
    log(f"  old tmp__ links removed: {removed}")

    count = 0
    for src in sorted(TMP_MODELS_DIR.iterdir(), key=lambda x: x.name.lower()):
        if src.name.startswith("."):
            continue
        if src.is_symlink() and not src.exists():
            log(f"  ⚠️ broken symlink skip: {src.name}")
            continue

        dst = PERSISTENT_MODELS_DIR / (prefix + src.name)
        i = 2
        while dst.exists() or dst.is_symlink():
            dst = PERSISTENT_MODELS_DIR / f"{prefix}{i}__{src.name}"
            i += 1

        os.symlink(str(src), str(dst))
        log(f"  🔗 {dst.name} -> {src}")
        count += 1

    log(f"  ✅ tmp model links: {count}")




# ============================================================
# Idle timeout handling
# ============================================================

def remove_flag_with_value(flags, flag):
    out = []
    skip = False
    for x in flags:
        if skip:
            skip = False
            continue
        if x == flag:
            skip = True
            continue
        out.append(x)
    return out

def apply_idle_timeout_flag():
    """
    MODEL_IDLE_TIMEOUT_MINUTES が None の場合は --idle-timeout を付けない。
    数値の場合だけ --idle-timeout <minutes> を起動フラグへ追加する。
    """
    global LAUNCH_FLAGS
    LAUNCH_FLAGS = remove_flag_with_value(LAUNCH_FLAGS, "--idle-timeout")

    if MODEL_IDLE_TIMEOUT_MINUTES is None:
        log("✅ model idle auto-unload: disabled")
        return

    minutes = int(MODEL_IDLE_TIMEOUT_MINUTES)
    if minutes <= 0:
        log("✅ model idle auto-unload: disabled")
        return

    LAUNCH_FLAGS += ["--idle-timeout", str(minutes)]
    log(f"✅ model idle auto-unload: {minutes} minutes")


# ============================================================
# GPU stale process handling
# ============================================================

def get_process_args(pid: int) -> str:
    r = run_capture(["ps", "-p", str(pid), "-o", "args="])
    if r.returncode == 0:
        return r.stdout.strip()
    return ""

def query_gpu_processes():
    """
    nvidia-smiからGPUを使っているプロセス一覧を取得する。
    戻り値: [(pid, process_name, used_memory_mib, args)]
    """
    r = run_capture([
        "nvidia-smi",
        "--query-compute-apps=pid,process_name,used_memory",
        "--format=csv,noheader,nounits",
    ])
    if r.returncode != 0 or not r.stdout.strip():
        return []

    procs = []
    for line in r.stdout.strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 3:
            continue
        try:
            pid = int(parts[0])
            used = int(parts[2])
        except ValueError:
            continue
        name = parts[1]
        args = get_process_args(pid)
        procs.append((pid, name, used, args))
    return procs

def is_textgen_gpu_process(pid: int, name: str, args: str) -> bool:
    """
    TextGen / llama.cpp由来らしいGPUプロセスだけを対象にする。
    Jupyter kernelや無関係な学習プロセスは極力殺さない。
    """
    current_pid = os.getpid()
    if pid == current_pid:
        return False

    hay = f"{name} {args}"

    patterns = [
        "/notebooks/textgen/server.py",
        "/notebooks/textgen",
        "llama_cpp_binaries",
        "llama-server",
        "llama-server-binary",
        "llama.cpp",
        "server.py --listen",
        "server.py --share",
    ]

    # ipykernelは殺さない
    if "ipykernel" in hay or "jupyter" in hay:
        return False

    return any(p in hay for p in patterns)

def cleanup_stale_gpu_textgen_processes():
    if not AUTO_KILL_STALE_GPU_TEXTGEN:
        log("ℹ️ AUTO_KILL_STALE_GPU_TEXTGEN=False: GPU stale process cleanup skipped")
        return

    log("\n🧹 GPU stale TextGen/llama.cpp process check")
    procs = query_gpu_processes()
    if not procs:
        log("✅ no GPU compute processes found")
        return

    targets = []
    log("  GPU processes:")
    for pid, name, used, args in procs:
        short_args = args[:180] if args else ""
        log(f"   - PID {pid} | {used} MiB | {name} | {short_args}")
        if is_textgen_gpu_process(pid, name, args):
            targets.append((pid, name, used, args))

    if not targets:
        log("✅ no stale TextGen/llama.cpp GPU process detected")
        return

    log("🧹 Terminating stale TextGen/llama.cpp GPU processes:")
    for pid, name, used, args in targets:
        log(f"  SIGTERM PID {pid} | {used} MiB | {name}")
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except Exception as e:
            log(f"  ⚠️ SIGTERM failed for {pid}: {e}")

    time.sleep(3)

    # 残っている対象だけSIGKILL
    remaining = []
    for pid, name, used, args in query_gpu_processes():
        if is_textgen_gpu_process(pid, name, args):
            remaining.append((pid, name, used, args))

    if remaining:
        log("⚠️ Some GPU processes survived SIGTERM. Killing:")
        for pid, name, used, args in remaining:
            log(f"  SIGKILL PID {pid} | {used} MiB | {name}")
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            except Exception as e:
                log(f"  ⚠️ SIGKILL failed for {pid}: {e}")

    time.sleep(1)

def show_vram_state(label="VRAM state"):
    if not SHOW_VRAM_AFTER_CLEANUP:
        return
    log(f"\n🧠 {label}")
    r = run_capture(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free", "--format=csv,noheader,nounits"])
    if r.returncode == 0 and r.stdout.strip():
        for line in r.stdout.strip().splitlines():
            name, total, used, free = [x.strip() for x in line.split(",")]
            log(f"  {name}: total={total} MiB used={used} MiB free={free} MiB")
    else:
        log("  ⚠️ unable to query VRAM state")

def get_gpu_memory_state():
    """
    GPUメモリ状態を取得。
    戻り値: [(name, total_mib, used_mib, free_mib)]
    """
    r = run_capture(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free", "--format=csv,noheader,nounits"])
    if r.returncode != 0 or not r.stdout.strip():
        return []

    out = []
    for line in r.stdout.strip().splitlines():
        try:
            name, total, used, free = [x.strip() for x in line.split(",")]
            out.append((name, int(total), int(used), int(free)))
        except Exception:
            pass
    return out

def min_free_vram_mib():
    states = get_gpu_memory_state()
    if not states:
        return None
    return min(free for _, _, _, free in states)

def kill_gpu_processes_aggressively(reason=""):
    """
    nvidia-smiに出ているGPU computeプロセスを強制停止する。
    Paperspace専用GPUでTextGenだけを使う前提の最終手段。
    """
    if not AGGRESSIVE_KILL_GPU_PROCESSES:
        log("ℹ️ AGGRESSIVE_KILL_GPU_PROCESSES=False: aggressive GPU cleanup skipped")
        return

    procs = query_gpu_processes()
    if not procs:
        log("✅ no GPU compute processes to kill")
        return

    log("\n🚨 Aggressive GPU process cleanup")
    if reason:
        log(f"  reason: {reason}")

    current_pid = os.getpid()
    targets = []

    for pid, name, used, args in procs:
        hay = f"{name} {args}"
        if pid == current_pid:
            continue
        if "ipykernel" in hay or "jupyter" in hay:
            # Jupyter自身は基本殺さない
            continue
        targets.append((pid, name, used, args))

    if not targets:
        log("✅ no safe aggressive targets found")
        return

    log("  targets:")
    for pid, name, used, args in targets:
        log(f"   - PID {pid} | {used} MiB | {name} | {args[:180]}")

    for pid, name, used, args in targets:
        try:
            log(f"  SIGTERM PID {pid}")
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except Exception as e:
            log(f"  ⚠️ SIGTERM failed for {pid}: {e}")

    time.sleep(3)

    remaining = []
    current_pids = {pid for pid, _, _, _ in targets}
    for pid, name, used, args in query_gpu_processes():
        if pid in current_pids:
            remaining.append((pid, name, used, args))

    if remaining:
        log("  Some targets survived. SIGKILL:")
        for pid, name, used, args in remaining:
            try:
                log(f"  SIGKILL PID {pid}")
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            except Exception as e:
                log(f"  ⚠️ SIGKILL failed for {pid}: {e}")

    time.sleep(2)

def ensure_enough_vram_before_launch():
    """
    モデルロード前のVRAM空きが異常に少ない時、古いGPUプロセスを掃除する。
    """
    log("\n🧪 VRAM preflight")
    show_vram_state("VRAM before hard preflight")

    free = min_free_vram_mib()
    if free is None:
        log("⚠️ could not determine free VRAM")
        return

    if free >= MIN_FREE_VRAM_BEFORE_LAUNCH_MIB:
        log(f"✅ free VRAM looks sufficient: {free} MiB")
        return

    log(f"⚠️ free VRAM too low before launch: {free} MiB < {MIN_FREE_VRAM_BEFORE_LAUNCH_MIB} MiB")
    log("   This usually means an old llama.cpp/TextGen process is still holding VRAM.")

    # まずTextGen系を掃除
    cleanup_stale_gpu_textgen_processes()
    time.sleep(1)

    free2 = min_free_vram_mib()
    show_vram_state("VRAM after targeted cleanup")
    if free2 is not None and free2 >= MIN_FREE_VRAM_BEFORE_LAUNCH_MIB:
        log(f"✅ free VRAM recovered: {free2} MiB")
        return

    # まだ少ない場合、専用GPU前提で強制掃除
    kill_gpu_processes_aggressively(
        reason=f"free VRAM still below threshold after targeted cleanup: {free2} MiB"
    )

    free3 = min_free_vram_mib()
    show_vram_state("VRAM after aggressive cleanup")
    if free3 is not None and free3 < MIN_FREE_VRAM_BEFORE_LAUNCH_MIB:
        raise RuntimeError(
            f"Free VRAM is still too low: {free3} MiB. "
            "Check nvidia-smi manually; another protected process may be using the GPU."
        )


# ============================================================
# Port / stale process handling
# ============================================================

def is_port_free(port: int, host: str = "0.0.0.0") -> bool:
    """指定ポートが空いているか確認する。"""
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind((host, port))
            return True
        except OSError:
            return False

def find_stale_textgen_pids():
    """
    古いTextGen server.pyプロセスを探す。
    自分自身のJupyter kernelは殺さない。
    対象は /notebooks/textgen/server.py または cwd付き server.py 起動に限定する。
    """
    current_pid = os.getpid()
    r = run_capture(["ps", "-eo", "pid,args"])
    if r.returncode != 0:
        return []

    pids = []
    for line in r.stdout.splitlines()[1:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            continue
        try:
            pid = int(parts[0])
        except ValueError:
            continue
        args = parts[1]

        if pid == current_pid:
            continue

        # notebook生成/検査中のプロセスやgrep系は除外
        if "grep" in args or "ipykernel" in args:
            continue

        # TextGen server.py だけ狙う
        looks_textgen = (
            "/notebooks/textgen/server.py" in args
            or ("server.py" in args and "--extensions" in args and "superboogav2" in args)
            or ("server.py" in args and "--listen" in args and "--share" in args)
        )

        if looks_textgen:
            pids.append((pid, args))

    return pids

def kill_stale_textgen_processes():
    if not AUTO_KILL_STALE_TEXTGEN:
        log("ℹ️ AUTO_KILL_STALE_TEXTGEN=False: stale process check skipped")
        return

    stale = find_stale_textgen_pids()
    if not stale:
        log("✅ stale TextGen process: none")
        return

    log("🧹 Stale TextGen process found. Terminating before launch:")
    for pid, args in stale:
        log(f"  PID {pid}: {args[:160]}")
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        except Exception as e:
            log(f"  ⚠️ SIGTERM failed for {pid}: {e}")

    time.sleep(3)

    # まだ残るものだけkill -9
    remaining = find_stale_textgen_pids()
    if remaining:
        log("⚠️ Some stale processes survived SIGTERM. Killing:")
        for pid, args in remaining:
            log(f"  kill -9 {pid}: {args[:160]}")
            try:
                os.kill(pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            except Exception as e:
                log(f"  ⚠️ SIGKILL failed for {pid}: {e}")

    time.sleep(1)

def get_flag_value(flags, flag, default=None):
    for i, x in enumerate(flags):
        if x == flag and i + 1 < len(flags):
            return flags[i + 1]
    return default

def set_flag_value(flags, flag, value):
    flags = list(flags)
    for i, x in enumerate(flags):
        if x == flag and i + 1 < len(flags):
            flags[i + 1] = str(value)
            return flags
    flags += [flag, str(value)]
    return flags

def prepare_launch_port():
    """
    7860が埋まっていたら、古いTextGenを止める。
    それでも空かない場合、7861〜7899の空きポートに逃がす。
    """
    global LAUNCH_FLAGS

    desired = int(get_flag_value(LAUNCH_FLAGS, "--listen-port", "7860"))

    log("\n🔌 Port check")
    kill_stale_textgen_processes()

    if is_port_free(desired):
        log(f"✅ port {desired} is free")
        return desired

    log(f"⚠️ port {desired} is still busy")

    if not AUTO_SELECT_FREE_PORT:
        raise RuntimeError(f"Port {desired} is busy. Stop old TextGen process or change --listen-port.")

    for port in range(PORT_RANGE_START, PORT_RANGE_END + 1):
        if is_port_free(port):
            LAUNCH_FLAGS = set_flag_value(LAUNCH_FLAGS, "--listen-port", port)
            os.environ["GRADIO_SERVER_PORT"] = str(port)
            log(f"✅ using fallback port: {port}")
            return port

    raise RuntimeError(f"No free port found in range {PORT_RANGE_START}-{PORT_RANGE_END}")


# ============================================================
# Launch
# ============================================================

def preflight_summary():
    log("\n✅ Preflight summary")
    critical = {
        "gradio": "import gradio, gradio_client",
        "ddgs": "from ddgs import DDGS",
        "llama_cpp_binaries": "import llama_cpp_binaries",
        "chromadb": "import chromadb",
    }
    bad = []
    for name, code in critical.items():
        if not module_ok(name, code):
            bad.append(name)
    if bad:
        raise RuntimeError("Critical modules still missing before launch: " + ", ".join(bad))
    log("✅ critical modules OK")

    if ENABLE_SUPERBOOGAV2 and FORCE_SUPERBOOGA_EMBEDDING_CPU:
        chroma_file = REPO_DIR / "extensions" / "superboogav2" / "chromadb.py"
        txt = chroma_file.read_text(encoding="utf-8")
        if "SUPERBOOGA_EMBEDDING_DEVICE" not in txt:
            raise RuntimeError("SuperboogaV2 CPU embedding patch is not active.")
        log("✅ SuperboogaV2 embedding device is patched to CPU/env-controlled")

def launch_textgen():
    log("\n🚀 Launching TextGen")
    env = os.environ.copy()
    env["HF_HOME"] = str(HF_HOME_DIR)
    env["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME_DIR / "hub")
    env["SENTENCE_TRANSFORMERS_HOME"] = str(ST_HOME_DIR)
    env["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTHONUNBUFFERED"] = "1"
    env["ANONYMIZED_TELEMETRY"] = "False"
    env["CHROMA_TELEMETRY"] = "False"
    env["POSTHOG_DISABLED"] = "true"
    env["NLTK_DATA"] = str((REPO_DIR / "extensions" / "superboogav2" / "nltk_data").resolve())
    env["SUPERBOOGAV2_EMBEDDING_MODEL"] = MULTILINGUAL_EMBEDDING_MODEL
    env["GRADIO_SERVER_PORT"] = str(get_flag_value(LAUNCH_FLAGS, "--listen-port", "7860"))
    env["SUPERBOOGA_EMBEDDING_DEVICE"] = SUPERBOOGA_EMBEDDING_DEVICE
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    cmd = [sys.executable, "server.py"] + LAUNCH_FLAGS + EXTRA_FLAGS
    log("▶ " + " ".join(shlex.quote(str(x)) for x in cmd))
    log("\nGradio share URLが出たら開いてください。停止はこのセルをInterrupt/停止。\n")

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO_DIR),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        log("\n🛑 Stopping TextGen...")
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()

# ============================================================
# Main
# ============================================================

log(f"=== {BOOTSTRAP_VERSION} ===")
print_feature_summary()
ensure_dirs()
setup_env()
check_gpu()
cleanup_stale_gpu_textgen_processes()
show_vram_state("VRAM after stale cleanup")
ensure_enough_vram_before_launch()
ensure_repo()
ensure_textgen_portable_deps()
ensure_gguf_loader_deps()
ensure_superbooga_deps()
configure_superbooga()
link_tmp_models()
apply_idle_timeout_flag()
prepare_launch_port()
log("✅ Distribution v1.0 distributable/polished logging active")
preflight_summary()
launch_textgen()


## トラブルシュート

### 放置後に `No model is loaded` と表示される

`MODEL_IDLE_TIMEOUT_MINUTES` に数値を指定している場合、その分数だけ無操作が続くとモデルがアンロードされます。

放置後もモデルを常駐させたい場合は、以下に戻してください。

```python
MODEL_IDLE_TIMEOUT_MINUTES = None
```

---

### `CUDA out of memory` が表示される

主な原因は以下です。

```text
大きいGGUFモデルでVRAMが埋まっている
カーネル再起動後も古いGPUプロセスが残っている
SuperboogaV2のEmbedding処理がGPUに乗ろうとしている
```

このNotebookでは、SuperboogaV2 EmbeddingはCPU固定です。

```python
FORCE_SUPERBOOGA_EMBEDDING_CPU = True
SUPERBOOGA_EMBEDDING_DEVICE = "cpu"
```

起動前からVRAMがほぼ埋まっている場合は、TextGen専用GPUであることを確認したうえで、必要時のみ以下を一時的にONにしてください。

```python
AGGRESSIVE_KILL_GPU_PROCESSES = True
```

---

### 7860番ポートが埋まっている

このNotebookは古いTextGenプロセスを検出し、必要に応じて停止します。  
また、7860番が使えない場合は7861〜7899の空きポートへ退避します。

```python
AUTO_KILL_STALE_TEXTGEN = True
AUTO_SELECT_FREE_PORT = True
```

---

### 依存関係のログが多い

通常時は長いTracebackを抑制しています。

```python
QUIET_MISSING_MODULE_CHECKS = True
PIP_QUIET_INSTALL = True
```

原因調査のために詳細ログを見たい場合は、一時的に以下に変更してください。

```python
QUIET_MISSING_MODULE_CHECKS = False
PIP_QUIET_INSTALL = False
```

---

### TextGen更新後に起動しない

まず以下を通常値に戻してください。

```python
UPDATE_TEXTGEN = False
FORCE_RESET_REPO = False
FORCE_REPAIR_DEPS = False
```

それでも解決しない場合のみ、次のようにリポジトリリセットと依存修復を行います。

```python
FORCE_RESET_REPO = True
FORCE_REPAIR_DEPS = True
```

実行後は、再び `False` に戻してください。

---

## 配布・再利用時の注意

- このNotebookはモデルファイルを含みません。
- モデルのライセンス・利用条件は、各モデルの配布元に従ってください。
- `--share` URLは秘密URLではありません。
- TextGen本体やSuperboogaV2の仕様変更により、将来パッチ箇所が変わる可能性があります。
- 起動セルは、必要に応じて `extensions/superboogav2/chromadb.py` をバックアップした上で一部パッチします。
- 重要な作業データやモデルファイルは、Notebook外にもバックアップを取ることを推奨します。
